# 11. Support Vector Machine lề cứng từ đầu (Hard-Margin SVM from Scratch)

## 1. Cơ sở lý thuyết Chapter 26
Hard-Margin SVM giả định dữ liệu có thể **phân tách tuyến tính hoàn hảo (Linearly Separable)**. Bài toán tối ưu hóa dạng nguyên thủy (Primal Objective):
$$\min_{w, b} \frac{1}{2} \|w\|^2 \quad \text{thỏa mãn} \quad y_i (w^T x_i + b) \ge 1, \quad \forall i$$
Trong đó nhãn mục tiêu được chuyển đổi: $0 \to -1$ và $1 \to +1$.
Độ rộng của lề (Margin width) là $\frac{2}{\|w\|}$. Các điểm nằm chính xác trên bờ lề $y_i (w^T x_i + b) = 1$ được gọi là các **Support Vectors**.

In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_utils import train_test_split
from src.preprocessing import HeartDiseasePreprocessor
from src.models.svm_hard_margin import HardMarginSVM
from src.metrics import calculate_metrics

# 1. Thử nghiệm trên Toy Dataset phân tách tuyến tính hoàn hảo để kiểm tra tính đúng đắn của thuật toán
np.random.seed(42)
X_toy_0 = np.random.randn(20, 2) - 3.0
X_toy_1 = np.random.randn(20, 2) + 3.0
X_toy = np.vstack([X_toy_0, X_toy_1])
y_toy = np.array([0] * 20 + [1] * 20)

svm_toy = HardMarginSVM(learning_rate=0.01, epochs=1000).fit(X_toy, y_toy)
print(f"Toy Data - Có phân tách tuyến tính không?: {svm_toy.is_separable_}")
print(f"Toy Data - Số lượng vi phạm lề: {svm_toy.n_violations_}")
print(f"Toy Data - Số lượng Support Vectors: {len(svm_toy.support_vectors_)}")

Toy Data - Có phân tách tuyến tính không?: True
Toy Data - Số lượng vi phạm lề: 0
Toy Data - Số lượng Support Vectors: 0


## 2. Thử nghiệm trên dữ liệu bệnh tim thực tế (Heart Disease Dataset)

In [2]:
df = pd.read_csv("../data/heart_cleveland_upload.csv")
X = df.drop("condition", axis=1)
y = df["condition"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = HeartDiseasePreprocessor()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

svm_real = HardMarginSVM(learning_rate=0.001, epochs=2000, penalty=1000.0).fit(X_train_proc, y_train)
print(f"Heart Data - Có phân tách tuyến tính hoàn hảo không?: {svm_real.is_separable_}")
print(f"Heart Data - Số lượng mẫu vi phạm ràng buộc lề cứng: {svm_real.n_violations_}")

y_pred = svm_real.predict(X_test_proc)
metrics = calculate_metrics(y_test, y_pred)
print("Kết quả trên tập kiểm thử:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Heart Data - Có phân tách tuyến tính hoàn hảo không?: False
Heart Data - Số lượng mẫu vi phạm ràng buộc lề cứng: 57
Kết quả trên tập kiểm thử:
  Accuracy: 0.7288135593220338
  Precision: 0.6410256410256411
  Recall: 0.9259259259259259
  F1_Score: 0.7575757575757577
  ROC_AUC: N/A


## 3. Thảo luận về giả định Hard-Margin
- Kết quả kiểm tra chỉ ra rõ ràng: `svm_real.is_separable_ = False`, và có nhiều mẫu vi phạm lề.
- Dữ liệu lâm sàng thực tế luôn chứa nhiễu, điểm ngoại lai (outliers) hoặc các bệnh nhân có chỉ số tương đồng nhưng kết quả chẩn đoán khác nhau. Vì vậy, giả định lề cứng không thỏa mãn trong thực tế, dẫn tới sự ra đời tất yếu của **Soft-Margin SVM** (Chapter 27).